# MGS-8 - Trouver l'Everest : relief reel et bassins d'attraction

**Serie MetaGeneticSharp** | Precedent : [MGS-7 - Landscape Explorer](MGS-7-LandscapeExplorer.ipynb)

MGS-7 explorait des paysages de fitness *synthetiques* (Sphere, Rastrigin...). Ici le
paysage est un **relief terrestre reel** : on cherche le point le plus haut - l'Everest -
sur quatre cartes d'altitude a zoom croissant, du globe entier a la region du sommet.

## Objectifs d'apprentissage

- Traiter une grille d'altitude (DEM) comme un **paysage de fitness a maximiser**.
- Visualiser le relief en **ASCII grayscale** (haute altitude = caractere dense), comme MGS-7.
- Lancer le **moteur MGS** (GA / WOA / Equilibrium Optimizer) sur ce relief, plus une
  **baseline PSO** externe, a budget d'evaluations egal.
- Mesurer le **taux de reussite** de chaque methode pour localiser le sommet, et relier ce
  taux au rapport **taille du bassin d'attraction / taille de l'espace de recherche**.

## Prerequis

- MGS-1 a MGS-5 (moteur, compounds geometriques WOA/EO), MGS-7 (heatmaps ASCII).
- Build du fork : `dotnet build c:/dev/MetaGeneticSharp/MetaGeneticSharp.sln`.

## Donnees

Reliefs **ETOPO1** (NOAA National Centers for Environmental Information, 1 arc-minute global
relief) downsamples a 40x40 par zoom et **embarques** dans le notebook (aucun appel reseau a
l'execution). A 1 arc-minute la maille moyenne l'altitude : le sommet de l'Everest y apparait
vers **8221 m** au zoom le plus fin (contre 8849 m officiels) - une attribution honnete de la
resolution de la source.

> **Note d'auteur (AUTHORSHIP).** Ce notebook est un *habillage pedagogique* : il consomme la
> librairie MGS (moteur, WhaleOptimisationAlgorithm, EquilibriumOptimizer, KnownFunctionGenes)
> sans la reimplementer. Le PSO est une baseline externe explicite (le bestiaire du fork n'a
> pas de PSO), au meme titre que le RandomSearchOptimizer de la lib.

*Duree estimee : 25-35 min.*

In [1]:
// MetaGeneticSharp + GeneticSharp DLLs from the fork build.
// Build prerequisite: dotnet build c:/dev/MetaGeneticSharp/MetaGeneticSharp.sln
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Extensions/bin/Debug/net9.0/MetaGeneticSharp.Extensions.dll"

using System.Linq;
using MetaGeneticSharp;
using GeneticSharp;

Console.WriteLine("Wiring OK : MetaGeneticSharp + GeneticSharp + Extensions.");
Console.WriteLine($"  WhaleOptimisationAlgorithm : {typeof(WhaleOptimisationAlgorithm).Name}");
Console.WriteLine($"  EquilibriumOptimizer       : {typeof(EquilibriumOptimizer).Name}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Wiring OK : MetaGeneticSharp + GeneticSharp + Extensions.


  WhaleOptimisationAlgorithm : WhaleOptimisationAlgorithm


  EquilibriumOptimizer       : EquilibriumOptimizer


## 1. Le relief reel comme paysage de fitness

Une grille d'altitude (Digital Elevation Model) est un tableau `N x N` de hauteurs en metres.
On la lit en **coordonnees normalisees** `(u, v)` dans `[0,1]^2` : `u` parcourt la longitude
(ouest -> est), `v` la latitude (sud -> nord). L'**altitude interpolee** (bilineaire) en `(u,v)`
est notre **fonction de fitness, a MAXIMISER** - l'inverse de MGS-5/MGS-7 ou un objectif
synthetique etait minimise. *Trouver l'Everest* = localiser le maximum global de la grille.

La classe `DemGrid` ci-dessous ne fait que stocker la grille et interpoler ; aucune logique
d'optimisation n'y vit.

In [2]:
// A downsampled real-elevation grid (ETOPO1, attributed in the header). Row-major:
//   index = row*N + col ; row = latitude band (0 = south), col = longitude (0 = west).
// Elevation in metres IS the fitness to MAXIMISE.
public sealed class DemGrid
{
    public string Name { get; }
    public double LatMin, LatMax, LonMin, LonMax;
    public int N;
    public int[] Elev;   // length N*N, metres
    public DemGrid(string name, double latMin, double latMax, double lonMin, double lonMax, int n, int[] elev)
    { Name = name; LatMin = latMin; LatMax = latMax; LonMin = lonMin; LonMax = lonMax; N = n; Elev = elev; }

    public double At(int row, int col) => Elev[row * N + col];

    // Bilinear interpolation at normalised (u,v) in [0,1]^2 : u -> longitude, v -> latitude.
    public double Interp(double u, double v)
    {
        u = Math.Clamp(u, 0.0, 1.0); v = Math.Clamp(v, 0.0, 1.0);
        double fx = u * (N - 1), fy = v * (N - 1);
        int x0 = (int)Math.Floor(fx), y0 = (int)Math.Floor(fy);
        int x1 = Math.Min(x0 + 1, N - 1), y1 = Math.Min(y0 + 1, N - 1);
        double tx = fx - x0, ty = fy - y0;
        double a = At(y0, x0) + (At(y0, x1) - At(y0, x0)) * tx;
        double b = At(y1, x0) + (At(y1, x1) - At(y1, x0)) * tx;
        return a + (b - a) * ty;
    }

    // Normalised location (u,v) of the highest grid cell -- the target of the search.
    public (double u, double v, double elev) GlobalMax()
    {
        int best = 0;
        for (int k = 1; k < Elev.Length; k++) if (Elev[k] > Elev[best]) best = k;
        int row = best / N, col = best % N;
        return ((double)col / (N - 1), (double)row / (N - 1), Elev[best]);
    }
}
Console.WriteLine("DemGrid defini (relief reel = paysage de fitness a MAXIMISER).");

DemGrid defini (relief reel = paysage de fitness a MAXIMISER).


In [3]:
// Embedded ETOPO1 relief grids (40x40 each), row-major lat south->north, lon west->east.
// Source: NOAA NCEI ETOPO1 1 arc-minute global relief, downsampled offline.
// World: bbox lat[-56.0,60.0] lon[-180.0,180.0], max=5102 m at row=29 col=29
var world = new DemGrid("World", -56.0, 60.0, -180.0, 180.0, 40, new int[] {
    -5078,-4936,-3906,-3843,-3507,-3295,-4206,-3294,-4279,-4536,-4696,-4494,-45,-4047,-4171,-3334,-3152,-3968,-5128,-2651,-3302,-5242,-2384,-5377,-4544,-5367,-4851,-4041,-2601,-4555,-3980,-3921,-4503,-4637,-3790,-3177,-3820,-4245,-5655,-5078,
    -5254,-5307,-4496,-4022,-3594,-4246,-3649,-3257,-4098,-4416,-4928,-4091,204,-390,-2464,-2261,-3310,-4201,-3571,-2061,-2849,-3748,-2749,-5538,-2760,-4605,-5438,-4129,-1827,-4441,-3925,-3665,-3756,-4682,-3604,-3105,-4228,-3752,-439,-5254,
    -4251,-5180,-4913,-4702,-4533,-4847,-3784,-3020,-3881,-4251,-4398,-3904,7,-211,-2354,-1563,-5003,-4613,-3806,-3011,-3257,-4284,-4503,-4054,-3908,-4516,-4600,-238,-3565,-4128,-3671,-3204,-4242,-3610,-3173,-3874,-4565,-3818,-554,-4251,
    -2096,-5240,-5287,-4560,-4988,-4819,-3755,-2957,-3750,-4314,-3996,-3743,997,-716,-6109,-5972,-5200,-4348,-2989,-3488,-4630,-4648,-4910,-4480,-3686,-3183,-4410,-725,-3499,-4066,-2789,-3517,-3796,-3715,-4227,-4253,-4803,-5313,-1204,-2096,
    -587,-4604,-5240,-5229,-5457,-4923,-4385,-3561,-3487,-4127,-3491,-3321,681,-117,-5585,-5096,-5082,-4270,-3781,-4273,-4354,-4667,-4861,-4876,-3450,-2977,-4809,-3801,-3480,-3401,-3350,-4122,-4182,-4481,-4601,-4884,-4641,-5021,434,-587,
    -2839,-4079,-5180,-5079,-5267,-5166,-4458,-3307,-3968,-3560,-3607,-3588,832,-89,-5543,-5209,-4813,-3936,-2762,-3924,-4773,-3076,-4660,-5042,-2542,-3740,-4431,-3949,-2866,-2798,-3847,-4322,-4724,-5156,-5251,-3086,-4895,-4736,-494,-2839,
    -3562,-3810,-5245,-5321,-4755,-5112,-4276,-3531,-3649,-3771,-3444,-4280,556,178,-4981,-5091,-4473,-3758,-3282,-4150,-5008,-5011,-5387,-4198,-4609,-2960,-5218,-4090,-2056,-3629,-4324,-4793,-5137,-5699,-5556,143,-4444,-4776,-1020,-3562,
    -3062,-5445,-5427,-5501,-5081,-4314,-3933,-3293,-3888,-2879,-4033,-4048,1362,45,-3131,-4819,-4645,-3900,-3525,-3257,-5243,-4702,-705,-3947,-5098,-3708,-5238,-4759,-2581,-3911,-4598,-5565,-1634,-4518,-1314,55,-4919,-2686,-1914,-3062,
    -2929,-5654,-5425,-5206,-5600,-4031,-3796,-3169,-3591,-3841,-3936,-3930,2252,65,-78,-3981,-3714,-3928,-4009,-3822,-5020,-3497,976,-3579,-4584,-4078,-3366,-4361,-3156,-1762,-1264,-5260,-39,139,20,70,96,-1546,-4157,-2929,
    -405,-5402,-5299,-5184,-4225,-4293,-3658,-2941,-3012,-3669,-3758,-3770,3706,60,789,-3822,-2620,-4663,-3540,-4444,-4951,-2078,1149,-1129,-4805,-4934,-4408,-4423,-4109,-2038,-3480,-5212,199,327,173,82,580,-1494,-2878,-405,
    -2592,-5599,-5028,-4599,-4221,-4089,-3676,-2725,-2685,-3789,-3543,-3893,3353,102,774,-2600,-4824,-4971,-3636,-4388,-2780,-378,1081,54,-4251,-5254,-5316,-3099,-3480,-2403,-4997,-5332,256,405,389,147,596,-1336,-3748,-2592,
    -2790,-5699,-4869,-4853,-4740,-3916,-3689,-2558,-3773,-3534,-4070,-4575,2193,136,469,-108,-4918,-5117,-2889,-5105,-3197,-160,1113,116,-3912,-4928,-4575,-3033,-4378,-3014,-5289,-4791,185,402,726,251,-280,-1407,-3638,-2790,
    -2882,-5559,-5019,-4339,-4487,-3195,-3966,-3323,-3743,-4047,-4231,-4632,2297,147,382,731,-4205,-5452,-3384,-5463,-5259,753,940,448,-2824,-4773,-4043,-2919,-4869,-2444,-5504,-5606,-46,332,317,261,-363,-1462,-3022,-2882,
    -2190,-5208,-4576,-3740,-2277,-4269,-3672,-3256,-3705,-4111,-4459,-2903,4093,402,678,819,-4559,-4736,-2485,-4384,-5483,740,994,1206,-1759,-4330,-1358,-4205,-4989,-2582,-5779,-5778,-3174,63,232,155,-877,-4415,-2951,-2190,
    -2410,37,-4785,-4607,-4370,-4122,-3874,-3348,-3625,-3694,-4385,-4337,2853,209,239,515,-2737,-4913,-1811,-4288,-5473,1021,1033,917,-2464,-3318,-196,-4238,-5186,-3190,-4447,-5330,-5675,-75,108,174,-4653,-4183,-3490,-2410,
    -2452,-4936,-2817,-4067,-4782,-4159,-3881,-3201,-3765,-3921,-4226,-1652,309,380,188,798,-4453,-5269,-3292,-4671,-5525,139,1168,954,-1682,-4492,-2968,-4126,-5145,-5258,-4266,-5865,-6201,-70,-23,-5,-1607,-5125,-3331,-2452,
    -4819,-4938,-3838,-4916,-4367,-4551,-4458,-4218,-3566,-3537,-3906,1503,205,110,322,423,-4957,-5154,-2779,-4358,-5348,252,933,861,-3284,-5045,-465,-3815,-5341,-4964,-5385,-3117,622,191,-124,-1,-3216,-4074,-4579,-4819,
    -5555,-5502,-5420,-5325,-4587,-4573,-4536,-4269,-3041,-3612,-3906,592,132,51,214,256,-4607,-5165,-3799,-5551,-5266,275,650,1145,-2963,-5179,-4092,-3364,-5168,-5039,-4852,-20,-45,-297,-3547,2358,-3022,-2288,-3396,-5555,
    -5473,-5388,-5197,-4706,-4406,-4590,-4634,-4041,-3693,-3326,-3149,3436,121,137,48,-23,-4348,-5202,-3759,-4902,-4637,586,464,1186,-2231,-5084,-4008,-3093,-4958,-4901,-4764,24,14,-5493,26,-2880,-2020,-2594,-4183,-5473,
    -5669,-5590,-5163,-4292,-4371,-4346,-4576,-3971,-3694,-3305,-2750,3188,134,111,13,-4124,-4086,-3672,-4441,-5027,-3999,550,469,1138,42,-5184,-4269,-3711,-4762,-4466,-5097,-47,75,-538,-3550,-3006,-4482,-3915,-4171,-5669,
    -5298,-5280,-4722,-4708,-4359,-4464,-4578,-3997,-3726,-3632,-1971,-1370,166,83,-11,-3200,-3719,-4149,-4693,-4184,-3032,718,511,783,478,-4912,-4802,-4039,-3388,-4107,-22,-74,1262,-4656,-3927,-3326,-4426,-3531,-4216,-5298,
    -5820,-6102,-2491,-4838,-5027,-4252,-4129,-4047,-3780,-3595,-3116,-3886,94,143,-3544,-4799,-3435,-1611,-4449,107,12,951,679,428,854,-4666,-2588,-4876,-2632,-3855,-1302,-47,-71,122,-1793,-2707,-4346,-4810,-4509,-5820,
    -5932,-5197,-5055,-5297,-5090,-4899,-4575,-4038,-3287,-3762,-3397,-32,178,-39,-4775,-3396,-4757,-4992,-9,280,108,216,579,396,1937,159,-3875,-4576,38,-3502,-280,1,-1386,-1829,-5005,-4438,-5002,-4691,-2784,-5932,
    -5151,-5883,-5529,-5359,-5225,-4622,-4718,-4173,-3451,-4425,-97,-3403,-1077,-2146,-5106,-5074,-5529,-4931,66,441,251,289,806,400,479,-2346,-4446,-4228,397,-3159,-366,30,-4307,33,-5117,-3283,-5945,-5424,-5271,-5151,
    -5421,-5163,-5443,-5636,-5279,-4829,-4460,-3796,-3438,-4516,698,-2245,-3931,-5183,-4401,-4951,-5429,-542,64,266,349,355,808,423,-68,880,-3852,-3671,208,-2779,-21,703,-3916,-3912,-4051,-1884,-5474,-4576,-5833,-5421,
    -3683,-5107,-5416,-5166,-5490,-5260,-4072,-3542,-2758,1233,-917,-1555,-183,-6043,-4186,-4674,-5158,-3611,56,308,387,477,648,363,149,428,-3687,-3385,503,-2308,726,-17,-3778,-5741,-5929,-1882,-5534,-2133,-1222,-3683,
    -3365,-4815,-4514,-5066,-5636,-5076,-4217,-3796,-203,-662,16,5,-5483,-5716,-4359,-4399,-4850,-4692,193,275,615,538,596,546,1192,178,-2974,-62,394,-15,947,7,-115,-5501,-5849,-3309,-5768,-5181,-5694,-3365,
    -5535,-4757,-4856,-5360,-4666,-4602,-4192,-3846,2002,-222,-610,-1,-5537,-5992,-4995,-4818,-6330,-4937,296,298,814,577,405,285,981,109,-2633,7,411,57,133,706,285,-795,-4792,-3413,-3758,-3624,-6082,-5535,
    -5357,-4694,-3830,-5501,-4834,-4233,-4385,-4191,1733,-42,-2794,-45,-5293,-5864,-4845,-4491,-4764,-4973,-1349,381,499,382,175,636,1109,-60,449,60,166,765,1249,1275,64,-103,-4790,-6383,-4972,-6024,-6071,-5357,
    -5276,-5311,-5653,-5142,-5068,-4454,-4444,490,1494,128,3,-802,-5277,-5335,-5684,-2812,-3954,-5258,-2894,670,233,449,85,153,516,763,1606,1158,1189,5102,4305,262,23,-59,-4402,-6199,-5991,-5985,-5309,-5276,
    -4859,-5886,-5758,-5488,-5427,-4352,-4192,74,1292,176,74,-26,-5334,-4948,-4666,-3297,-3530,-5398,-4419,1695,431,-262,-2043,-1588,426,2211,1972,2992,4884,4952,4693,1783,37,-77,-659,-5446,-5933,-5262,-5710,-4859,
    -4993,-5635,-5972,-5768,-5472,-5407,-4198,1308,1877,321,199,103,-4575,-5014,-5297,-4191,-2286,-4540,-4557,-1010,1020,-498,-39,-114,403,2033,862,1494,4453,5099,2944,1610,44,-67,-238,-7001,-5828,-4700,-4986,-4993,
    -5385,-5143,-5412,-5435,-5439,-4739,-3270,2537,3731,351,153,365,-2973,-5090,-5394,-5004,-2014,-3784,-3716,559,-2192,-3481,-22,897,1913,-931,119,2809,1139,837,3702,1798,88,-20,-1993,-2025,-5659,-5237,-6368,-5385,
    -5984,-5979,-5692,-5266,-4505,-3972,-328,1649,2093,465,163,595,-191,-4355,-3037,-4795,-3244,-3760,-5341,808,-2277,344,585,-2167,-307,-502,80,435,4283,1406,1902,957,1464,429,-2939,311,-5162,-5563,-4624,-5984,
    -5633,-5808,-5331,-5259,-4627,-3911,-436,2262,1171,573,192,408,148,-135,-18,-4619,-2970,-3522,-4546,-4595,761,271,471,-62,194,-33,29,229,595,568,2447,1197,1218,130,394,-128,-8533,-5878,-4021,-5633,
    -5146,-5432,-5118,-4914,-4383,-3512,18,960,666,249,-54,342,1,-475,-138,-4153,-3651,-4126,-4646,-36,273,396,133,114,127,-17,272,338,674,1607,2404,1373,674,217,42,-69,-3266,-5523,-5584,-5146,
    -3781,-7162,-4790,-4881,-3802,-3182,1997,1532,612,216,302,89,565,269,-322,-4315,-3385,-3954,-661,-48,12,254,180,133,86,63,337,367,174,1501,1323,855,730,485,1166,292,-1061,-5492,-4223,-3781,
    -277,-1904,-669,-4586,-3844,-1090,768,785,524,238,129,81,543,337,-3277,-3690,-2743,-3803,-1892,-4,-41,-1,135,217,154,136,376,118,110,337,2013,673,923,544,602,-37,-489,-3232,-3865,-277,
    -3790,-92,-82,-76,-3704,237,1452,514,581,261,-84,-6,337,-116,-3591,-3388,-2270,-3157,-204,549,-67,158,82,255,118,124,411,85,136,132,216,541,1625,931,742,-123,-477,205,-1590,-3790,
    -2641,-76,27,-49,-130,685,722,304,458,201,-197,-79,-23,-1751,-3211,-1892,-2349,-2306,-1340,-233,-249,323,15,24,167,166,237,36,50,150,370,392,589,512,175,416,982,217,-113,-2641
});

// TibetanPlateau: bbox lat[27.0,45.0] lon[70.0,100.0], max=6209 m at row=18 col=14
var tibetanPlateau = new DemGrid("TibetanPlateau", 27.0, 45.0, 70.0, 100.0, 40, new int[] {
    72,157,208,276,288,337,397,406,347,235,171,146,152,125,123,105,104,84,91,78,96,319,1488,1893,378,707,1960,1488,1399,1437,276,87,93,317,474,1886,362,3182,2035,2741,
    80,134,149,181,292,300,344,458,461,224,177,172,157,143,135,115,104,95,108,596,1727,1692,3408,1601,2323,3549,2798,3190,1695,2184,1328,1657,98,123,224,2843,1514,2394,3437,3391,
    73,124,110,148,217,287,322,381,331,319,187,180,165,158,147,139,272,1409,676,1203,1168,4459,5352,5312,5104,4519,3480,5058,4583,5180,1652,1569,1051,132,214,2047,1532,3550,2851,3927,
    76,88,110,121,186,216,271,279,248,234,204,184,170,179,162,660,1699,2142,2278,4290,3259,4982,4623,5337,4565,4730,5482,4149,5197,4438,3387,1079,1362,1808,2074,3494,4195,2082,2862,4104,
    89,89,108,125,152,200,222,218,217,220,218,212,190,216,538,907,1958,5022,3874,4843,5131,5036,5325,5620,4755,4776,4857,4479,4745,5028,4095,4431,1609,3381,4112,4567,4562,3369,3823,4708,
    302,98,112,135,150,177,194,212,221,233,237,235,375,1407,745,3156,2877,4609,5880,4752,4625,4902,4751,4744,4533,3997,4782,3591,3564,5155,4979,4732,4474,2115,4425,4956,5234,4201,3483,4426,
    589,110,115,131,156,173,192,214,231,252,258,449,936,1388,2319,4960,4113,4912,4755,5608,5738,5223,5058,4647,5269,4326,4586,4102,4409,5034,5210,3670,4332,3285,3865,5203,4130,4408,4619,4763,
    1201,122,123,139,155,171,196,221,247,271,349,1635,1396,3571,4626,4155,5407,4844,5119,5369,5686,5217,5083,5165,5380,5307,5051,4636,5250,5111,4242,3858,3994,3102,5015,3442,5024,4495,3304,4661,
    1175,138,138,143,168,176,195,228,264,449,1783,1777,3891,4068,4729,4585,5758,5094,5618,5166,4881,4721,5466,5032,5254,4940,4979,4724,4555,4932,4861,4930,5212,5233,4236,4461,3746,4354,4238,4776,
    1450,150,152,152,176,196,209,221,257,1157,2073,4274,5123,4495,4572,5704,5530,5414,4805,4818,4951,4834,5017,4765,4998,4686,4970,4721,4747,5033,4989,5100,4066,3989,4380,4037,4672,4629,4389,4435,
    1710,168,172,163,188,203,220,240,420,815,3346,3279,4670,5288,5536,5331,4822,4716,5694,4819,5343,4843,4947,4856,4620,4554,4860,4826,4779,4572,4614,4897,4209,4613,4760,4611,4611,4454,4572,3363,
    649,198,191,172,191,212,225,254,609,3416,5258,3843,4590,4327,4729,5167,5027,5421,4689,4768,4733,4684,4616,4774,4714,4644,4775,4572,4588,4846,4944,5019,5058,5158,4301,3960,4491,4180,4458,3888,
    1337,306,203,821,202,219,255,591,1485,4832,5449,5303,5850,4498,4931,4810,4379,4827,4865,5029,5020,4819,5095,5145,4934,4750,4754,4776,4860,5076,5245,5076,4655,4407,3939,4688,3689,4339,4372,4488,
    1050,346,361,427,457,241,545,1868,3860,4749,5134,5938,4214,5727,4739,4791,4584,4558,4573,5093,4747,4860,4761,5072,5000,5155,5064,5080,5039,5362,4795,4730,4944,4892,4512,4199,4584,4432,4649,4243,
    1130,938,428,411,452,657,3717,2758,5219,4436,5409,5710,5244,4426,4967,5123,5469,5544,4874,5183,5682,4946,4807,5584,5142,5191,4973,5355,5473,5358,4706,4759,4699,5053,4524,4886,4611,4507,4716,4331,
    1851,1946,386,284,1123,1613,2181,4109,5016,4171,4071,4255,4905,5576,4970,5203,5255,5017,4989,5052,4984,5027,5128,5139,5170,5952,5012,5551,5164,4650,4559,4570,4949,5084,4949,4651,4720,4492,4339,4033,
    1694,538,774,465,922,1910,1575,4419,4619,4253,4354,6050,5699,5974,5604,5738,5311,5423,4988,5332,5122,5220,4959,4967,4929,4841,5351,4921,4681,4914,4565,4491,4500,4715,4672,4640,4569,4235,4448,4549,
    2212,2571,997,1222,1783,4166,3321,4263,3336,3976,5547,5481,5218,5218,5187,4967,5492,5050,5068,5279,5227,4945,5051,5016,5147,4939,4986,5069,4951,4867,4550,4545,4889,4760,4646,4408,4267,4260,4627,4155,
    4416,2786,1320,3981,3067,2911,4310,4003,3122,5156,4912,5460,5129,5098,6209,5918,5246,4893,4948,4953,5101,4892,4984,4987,5236,4787,4810,4913,5174,5030,4634,4449,4745,4552,4671,4586,4586,4122,4159,3370,
    4701,3857,4059,5008,3995,4184,1799,2883,4447,5693,4933,5443,5519,4811,5091,4722,5297,5110,5036,5047,5010,5056,5178,5127,5130,4998,4883,5206,5136,4696,4807,4949,4386,3987,4763,4649,4675,3942,4555,3920,
    2106,3358,3960,3814,4525,4063,4229,5816,5437,4612,5033,4053,4717,3230,2776,2611,4726,5124,5018,4822,5124,5097,5216,5143,5027,5015,4948,4915,5114,3972,5048,3872,3286,3739,3528,2837,2923,3851,4652,2947,
    2663,3379,2973,4747,5404,5018,5109,5039,3764,4382,5302,5317,2482,2567,1825,1601,2054,2460,3834,5336,4754,4889,4902,5068,4770,4891,4524,5010,3989,3762,3476,2760,2714,2686,2690,2772,3044,3978,3082,3194,
    1919,2391,3962,4809,5004,5012,5001,4241,3731,3062,2909,2053,1667,1341,1319,1341,1413,1414,1838,2623,3726,4845,4168,4483,4263,4267,4136,4255,4335,3177,2817,2702,2684,2731,3448,2807,3144,3753,3562,3194,
    1159,3480,2900,4935,4004,4212,4411,3302,3316,2421,1697,1336,1283,1277,1264,1292,1334,1312,1352,1493,1856,2973,3224,3868,4576,3920,4032,4881,3254,2943,2727,2703,2901,3206,3829,3642,3794,4244,4101,3640,
    1237,3917,4237,3980,5069,4101,3847,4408,2777,1480,1269,1262,1242,1235,1214,1225,1204,1240,1243,1250,1281,1384,2754,3387,3970,3557,3309,2876,3147,2720,2749,2792,2915,4677,4504,4329,4219,4191,4029,4224,
    1905,3053,4697,5244,3694,4470,4320,4107,2506,1226,1195,1209,1218,1207,1260,1173,1157,1173,1170,1189,1156,1086,1093,1322,2799,3438,2935,3912,2726,2719,2734,3107,3298,4057,4041,4400,4964,4071,4248,2913,
    2217,3194,4301,3490,4699,4682,3778,2460,1284,1191,1170,1164,1149,1157,1137,1136,1115,1124,1102,1089,1094,1062,995,852,897,1788,3106,3347,3655,2898,2832,3247,3071,3844,3380,3952,4574,4344,4130,1890,
    3406,4480,3872,3043,4786,4905,3832,1869,1263,1192,1150,1127,1105,1130,1103,1094,1074,1065,1068,1029,1014,994,942,856,807,802,1142,1560,1728,1741,2268,2485,2899,3066,3776,2764,3609,2306,1453,1466,
    2075,1893,2696,2940,3018,3573,3535,3494,1980,1532,2024,1118,1089,1068,1059,1054,1032,1010,997,977,981,947,879,874,821,802,795,793,1055,1173,1202,1276,1663,1954,2280,2280,1812,1447,1282,1458,
    629,423,459,1209,2061,3219,3072,3336,3939,2450,2005,1919,1077,1049,1029,1017,996,970,942,928,920,907,903,858,830,811,791,790,1018,820,864,1022,1062,1115,1303,1460,1241,1369,1331,1147,
    1168,669,391,457,966,2272,3156,3296,3222,3002,2577,3069,2640,1066,1046,998,981,959,942,927,909,898,869,850,918,1049,923,792,887,1280,1190,1770,1574,1447,1573,1705,1601,1546,1632,1091,
    2252,3040,1035,1064,1307,1742,1913,2075,2715,3664,4449,3528,1384,1185,1176,996,982,971,939,924,904,887,957,1948,1488,1487,1425,1081,881,1120,1108,1262,1521,1857,2031,1961,1529,1280,1454,1035,
    2022,2195,2544,2567,965,2488,3606,3033,3276,3399,4188,3740,3768,4097,2159,1524,1206,1124,971,931,899,954,1063,1107,1269,779,1172,1338,1180,1104,1123,1140,1497,1891,2150,1813,1474,1496,1048,977,
    1019,3185,2511,2879,3191,2179,3539,2158,2437,1601,1602,3817,3140,3776,3916,3501,2468,1863,2519,2459,3345,1109,1078,1496,1013,1119,800,963,1040,671,595,714,988,1373,1790,1820,1486,1267,948,902,
    730,1054,925,2648,2824,1045,1916,1283,2366,2713,1601,1699,2561,2075,2066,3310,2867,3946,2550,2797,3682,2093,3140,2855,214,-152,327,481,393,99,551,780,1484,1385,1411,1169,1102,1266,1201,1125,
    958,438,481,623,586,591,611,2056,1000,1394,2952,1476,1816,2142,2276,1705,2069,2096,3530,3193,3316,3435,3846,2409,1064,815,1076,702,808,902,1171,1692,2429,979,1002,1569,1542,1427,1036,1493,
    417,382,423,495,557,465,1222,805,605,588,532,637,766,697,1031,987,905,1699,2491,2693,3864,3344,1653,1118,2331,2513,1995,2064,1433,2099,1627,3009,885,431,1022,1033,855,909,809,1066,
    316,341,374,385,388,460,779,656,553,595,752,1295,876,560,627,1106,2562,3038,3332,2607,1117,911,628,550,908,656,795,1013,1580,1813,1430,932,542,803,1197,1280,1371,1372,1444,1369,
    336,346,310,306,338,513,418,387,427,517,905,1704,1737,2724,2238,2859,373,1119,416,376,394,384,406,433,548,549,597,884,973,1178,738,604,1083,1888,1614,1669,1302,1452,2234,1751,
    211,281,326,302,495,419,351,368,386,395,469,604,2693,3226,1662,859,622,252,256,285,320,343,432,530,685,832,1010,1149,948,2220,2293,1899,1474,1892,1177,1410,2401,2521,1822,2015
});

// HimalayaArc: bbox lat[27.0,30.5] lon[83.0,89.0], max=6558 m at row=8 col=33
var himalayaArc = new DemGrid("HimalayaArc", 27.0, 30.5, 83.0, 89.0, 40, new int[] {
    83,80,83,84,88,89,89,86,84,82,80,79,85,93,96,95,99,122,188,265,311,200,345,864,711,807,1784,657,1410,988,1073,1640,1881,2232,1963,784,1436,699,496,1187,
    84,82,89,86,92,92,94,91,89,87,84,90,91,108,116,126,123,163,369,458,780,825,1184,750,536,1912,1249,1136,921,1336,923,1879,2694,2110,824,561,1253,2087,1419,1646,
    85,83,86,97,94,97,95,94,103,96,94,107,107,164,169,185,454,310,543,598,1040,1414,561,1053,903,1266,1170,1365,1005,2474,908,1254,1796,2466,1264,1856,512,1669,1476,2719,
    88,86,88,99,99,105,100,108,119,114,106,159,226,305,568,479,235,294,475,952,724,1026,1422,1422,821,2155,1868,617,857,2284,2215,1109,1699,2204,990,1406,1155,1840,2924,2764,
    90,90,92,96,104,103,107,177,214,400,310,274,418,698,507,453,429,656,1339,1000,852,862,2233,2282,1097,1585,1569,842,1005,2959,878,1689,2191,3066,1591,2461,2147,2588,4071,3244,
    93,95,97,101,103,104,122,258,346,205,611,505,537,595,970,805,1634,1863,873,1305,1183,1637,2680,2233,1384,2914,2530,899,1201,3316,1321,2845,2246,4235,3729,2581,1609,2528,4131,3652,
    100,103,104,110,113,118,171,168,157,217,210,276,560,1153,1625,1918,2308,1501,893,1885,994,2664,2306,2798,1926,2139,2940,2099,1448,3871,2119,2576,3571,5149,5046,3311,1346,2697,3872,3886,
    112,117,116,132,154,739,315,200,160,194,206,514,1063,1796,1679,1351,1743,928,1852,1916,1130,2007,3423,3560,2235,4475,4060,3300,2449,3025,4367,3908,3656,5094,4667,2807,3978,2469,4711,4745,
    134,153,214,393,917,787,621,623,512,278,284,1475,1236,1764,1010,1310,1456,876,896,2153,1933,2227,3740,4808,3460,5195,5457,3875,2372,3318,5267,4665,5378,6558,5292,4874,3337,4662,4874,4688,
    978,1033,1343,1300,864,1021,704,719,720,943,847,574,520,603,1529,1847,1978,1467,1005,1710,2090,1657,4574,5825,3616,5469,5990,4972,3809,2925,5240,5322,5508,5541,5729,4714,3983,4116,4948,4534,
    983,1795,1002,1080,736,465,621,547,899,659,496,1117,736,1190,607,1174,1933,1196,1347,1636,3812,2462,4186,5876,5053,4654,5562,4923,3735,3512,4569,5576,5468,6218,5278,4875,4131,4988,5303,4842,
    819,1138,937,787,1262,792,839,549,491,747,574,694,859,1733,1058,2039,2335,2045,2697,3503,4489,2958,4573,5291,4982,5349,5422,4499,4580,4569,4943,5250,5575,6108,5716,5624,5583,5886,5744,4982,
    1382,1440,1361,1944,1669,1163,1237,646,462,765,983,1018,851,1198,2667,3061,4579,4029,3174,4480,5226,4739,4256,5173,6152,5860,6256,5108,4009,4667,5770,4918,5528,5179,4938,5108,5627,5281,5231,5084,
    1321,1094,1162,2273,1169,1265,1282,707,610,1041,1535,1133,2051,2157,3286,2277,4077,5409,5745,4546,5192,4964,5010,5555,6411,5355,5606,5273,4339,5225,5544,4814,4503,4387,4610,4785,4794,5095,4933,4928,
    2061,1696,2343,2180,978,1477,1279,1010,1301,1138,2064,2974,2392,4112,3879,3471,5394,5694,5235,5235,5189,5317,5408,5716,5949,5326,4757,5182,4792,4752,5863,4437,5005,4503,4394,4502,4671,4977,4815,4646,
    2964,1453,3097,1846,1962,2235,2018,1931,2568,1790,3895,4909,3080,4507,5191,4045,3278,5265,6512,5271,4716,5649,6175,5255,4930,5121,4890,4650,4645,4496,4742,4323,4320,4634,4686,4676,4653,5049,5092,5093,
    2548,2835,2169,2489,1734,3216,3778,3325,4522,2006,3671,4712,2777,4129,4074,3798,4360,5686,5828,4932,4889,5058,5193,4993,4720,4958,4608,4315,5288,5078,4192,4386,4570,4758,4446,4462,4576,4772,5056,5115,
    3212,3534,2557,3087,2721,3773,4172,6238,3911,2881,6016,3870,2678,4364,5201,4968,4577,5482,5303,5496,5056,5167,5372,4410,4665,4802,5034,5161,5124,4188,4726,4230,4544,4308,4632,4754,4903,5299,5610,4770,
    3240,4092,3790,4740,2931,4107,6008,4341,3751,4341,5415,5782,3898,5458,4686,5018,5570,5080,5123,4742,4993,5353,4960,4500,4332,4402,4338,4312,4383,4277,4598,4596,4291,5001,4924,5459,4997,5677,5615,4678,
    3687,5157,5621,5390,3418,4771,4931,4927,4917,5764,5395,4534,4933,4492,4937,4276,5423,4680,4734,4744,5180,4910,4374,5252,4578,4777,4480,4766,5232,5585,5598,4658,5575,5495,5396,5640,5255,4963,5335,4668,
    5106,5303,5050,5414,5427,3080,5435,5205,5422,5599,5094,5416,4867,5268,5299,4162,5233,4584,4634,4764,5163,4593,4373,5470,4515,4711,4939,5102,5025,5627,5720,5060,4856,5605,5187,5271,5058,5050,4727,4809,
    2897,3664,4369,5521,4852,3264,4548,5837,5667,6305,5880,5534,5507,5313,4624,4313,4928,4580,5452,5501,5333,4741,4415,5259,5218,5036,5453,5312,5171,5254,5109,4797,4508,4633,5367,4489,4385,4569,4437,4571,
    4807,5228,5170,5297,5668,5284,3823,4553,5430,5587,5142,5063,5040,5703,5530,5385,4732,4597,5549,5333,5498,4876,4453,5250,5143,5068,5287,5493,5202,5090,4521,4670,4254,5019,4599,4219,4291,4630,4745,4491,
    4323,4212,4707,4652,5651,5282,3614,4884,5409,5212,4949,4790,5507,4986,4714,5054,4677,5009,4743,4732,5319,5434,4789,5504,5317,5115,5182,4950,4811,4753,4406,4051,4279,4467,4979,4518,4212,4080,4416,4533,
    4566,4509,4907,5301,5566,5937,4080,4286,5040,4860,4909,4743,5204,4533,4563,4937,4899,5124,4922,4795,4694,5275,5625,5237,4989,4448,4617,4398,4988,4505,4249,4129,4641,4247,4362,3942,3985,3934,4203,3862,
    4429,5141,5310,5460,5545,5694,4302,4872,5516,4774,4548,4540,4508,4564,5054,4723,4482,4826,4578,4682,4945,4571,5034,4388,4859,4944,4866,4758,4845,4810,4386,4154,4498,4343,4151,4140,4697,4300,3918,4212,
    4790,4804,4516,5493,5426,5767,4833,4854,4867,4764,4608,4958,4725,4671,4586,4491,4607,4877,5062,5064,5331,4745,5625,5011,5138,4689,4553,4325,4480,4597,4688,4180,4078,4437,4117,4042,3898,3881,3835,3829,
    5212,4369,5243,5672,5052,5274,4699,4751,5147,5060,4563,4793,5094,4677,5047,4866,5235,5089,4996,5054,5172,4834,5201,4768,4584,4724,5038,4513,4657,5018,4306,4101,4138,4793,4031,5364,4291,4957,4180,4663,
    4295,4694,5414,4986,5026,5361,4744,4662,4657,4616,4752,5224,5036,4715,5001,5068,5342,5030,4980,5066,4953,4769,4848,4651,4920,4779,5093,5055,5489,5520,4898,5129,5235,5089,4307,5284,4172,5386,5309,5137,
    4926,5338,5521,4763,4868,4837,4750,4551,4736,4784,4769,4709,5574,5399,4939,5090,5443,5239,5096,5522,5638,4895,5644,5563,5037,5545,5192,5484,5446,5512,5130,5132,5563,5125,4587,5245,4974,5066,5448,4614,
    5611,5257,4654,4715,4745,4692,5324,4561,4868,5007,5453,5670,4942,5338,5607,5525,5403,5542,5415,5715,5785,4906,5632,5630,5483,5837,4932,5779,5190,5604,4557,5121,5088,4957,5339,4787,4662,4544,5487,5038,
    5154,4752,4858,4741,4579,4934,4721,4922,4936,4882,5235,5542,5246,5631,5356,5404,5744,5675,5226,5819,5526,4953,5467,5623,5374,5488,5353,5125,5577,5012,4545,5387,5182,5489,5562,4851,4826,5474,5315,5179,
    4930,4876,4765,4623,4574,4699,5175,4686,4777,5091,5738,5323,5295,5499,5637,5676,5718,5453,5399,5645,5685,5438,5200,5270,5558,5272,5261,5291,5510,5114,4727,4790,4590,5034,5047,5325,5142,4910,5112,4707,
    4927,4621,4690,4578,4730,4936,4753,4832,5020,5602,5853,5720,5840,5931,5491,5454,5571,5293,5818,5289,5452,5508,5645,5034,5460,5264,5321,5486,5163,5187,4921,5065,5184,4976,5115,4892,5409,5109,5401,4496,
    4613,4595,4586,4620,5054,4803,4650,4830,5485,5397,5054,5736,5481,5546,5230,5680,5650,5305,5811,5145,5418,5666,5007,5003,5200,5766,5368,5258,5247,5361,5017,5443,5646,5361,5329,4873,5480,4686,4755,5218,
    4734,4603,4601,4774,4726,4696,5035,5371,5262,5845,4949,5762,5198,5610,5792,5781,5574,5372,5163,5355,5163,5910,4988,5195,5249,5597,5119,5468,5588,5245,4766,5085,5441,5160,5434,5106,5322,4920,4966,5230,
    4629,4925,4635,4810,4722,5044,5227,5505,5383,5336,4989,5645,5443,5870,5270,5428,5406,5439,5220,4964,5123,5634,4715,5231,5458,5041,4987,5322,4957,5168,5046,4923,5040,5021,5457,5522,4904,5459,5445,5311,
    4668,5120,4740,4652,4991,5482,5730,5473,5393,5111,5555,5457,5601,5548,5157,5152,5106,5131,5564,4912,5521,5643,4714,4803,5009,5156,5136,5622,5600,5422,5599,5622,5204,5115,5052,5587,4838,5444,5330,5274,
    4988,4923,4818,4808,5486,5423,5811,5387,5708,4979,5788,5736,5645,5284,5468,5117,5029,4916,5184,5036,5101,5166,5265,4974,5088,5196,4961,4968,5209,5569,5568,5369,5449,5323,5362,5282,4823,5158,5057,4966,
    5715,5051,5384,5427,5454,5567,5562,5644,5978,4853,5616,5848,5224,5187,4873,5129,4824,4803,4822,4856,5066,5024,5038,5849,4937,5154,5575,4985,5151,5123,5261,5347,5447,5163,5271,5289,5374,4809,5257,4893
});

// EverestRegion: bbox lat[27.7,28.3] lon[86.6,87.2], max=8221 m at row=13 col=33
var everestRegion = new DemGrid("EverestRegion", 27.7, 28.3, 86.6, 87.2, 40, new int[] {
    4643,4805,4898,4648,4049,3849,3405,2827,2937,3648,4244,4844,5254,4804,4199,4606,5246,5497,5598,5696,5494,5006,4556,4337,4725,5112,5139,4703,4335,4193,3945,4076,4178,4362,4516,4118,3553,3538,4138,4345,
    4778,5147,5129,4604,4146,3960,3469,2888,2905,3406,3802,4327,5064,5153,4542,4363,5249,5575,5722,5585,5257,4906,4648,4749,4844,4978,5425,5081,4539,4089,4125,4379,4326,4102,3894,3691,3799,4273,4319,4132,
    5152,5740,5187,4560,4190,3818,3316,2881,2789,3054,3470,4034,4813,5214,4848,4446,4651,4822,5121,5290,5150,4862,4799,5153,5382,5298,5539,5213,4869,4628,4541,4295,4157,4170,4214,4323,4426,4321,3919,3723,
    5594,5361,4769,4317,4252,3947,3403,2982,2901,3128,3532,4118,4790,5377,5338,5130,5025,4983,5073,5281,5278,5012,4891,5111,5378,5575,5313,5049,4883,4764,4534,4480,4706,4894,4928,4851,4471,3991,3928,4162,
    5551,5186,4729,4616,4615,4203,3709,3253,3001,3219,3692,4335,5040,5677,5929,5788,5548,5206,5122,5345,5474,5252,5018,5302,5951,6114,5298,5217,5020,4853,4736,5035,5285,5276,5154,4760,4084,4000,4306,4483,
    5231,5077,5014,5213,4922,4406,4017,3528,3163,3377,3882,4589,5381,6075,6165,5762,5469,5231,5282,5498,5501,5250,5103,5323,5844,6381,5822,5481,5298,5194,5110,5159,5280,5233,4970,4540,4096,4293,4674,4656,
    5447,5239,5190,5275,4857,4275,3825,3460,3287,3515,4053,4794,5641,6398,5894,5563,5449,5514,5504,5532,5471,5299,5238,5310,5568,6006,6031,5777,5740,5428,5090,4879,4877,4870,4708,4515,4492,4764,4847,4637,
    5201,4985,4797,4666,4375,3902,3616,3538,3464,3555,4069,4668,5275,5739,5241,5404,5469,5628,5666,5627,5546,5417,5339,5407,5608,5812,5990,5956,5887,5555,5008,4800,4824,4903,4864,4759,4972,5200,4873,4591,
    4627,4584,4372,4091,3965,3829,3858,3959,3823,3628,3839,4204,4573,4646,4775,5092,5224,5344,5625,5726,5602,5433,5367,5515,5715,6054,6076,5968,5867,5759,5158,4966,5115,5315,5332,5185,5264,5249,4755,4466,
    4802,4754,4423,4028,4137,4313,4307,4356,4271,3852,3695,3957,4213,4318,4702,4786,5077,5252,5536,5667,5570,5439,5417,5536,5711,6090,6116,5985,6000,5832,5269,5071,5279,5489,5507,5383,5163,4862,4550,4271,
    4781,4644,4319,4190,4595,4842,4763,4717,4524,4076,4070,4314,4113,4235,4472,4768,5210,5412,5597,5687,5638,5544,5504,5577,5699,6004,6058,5962,5776,5567,5359,5308,5581,5723,5454,5111,4881,4613,4589,4504,
    4470,4457,4262,4349,4792,4978,5034,4795,4306,4170,4517,4786,4629,4355,4329,4770,5091,5255,5231,5437,5632,5579,5624,5849,5911,6133,5920,5686,5475,5371,5521,5900,6361,6471,5900,5150,5014,5019,4924,4865,
    4783,4757,4402,4505,5101,5211,5084,4749,4359,4502,5005,5104,5130,4857,4380,4535,4739,4905,4972,5160,5379,5371,5262,5448,5882,6017,5776,5456,5574,5635,5905,6684,7640,7804,6783,5646,5458,5215,4653,4492,
    5013,5013,4476,4602,5173,5329,5047,4629,4507,4842,5159,5180,4937,4878,4435,4613,4637,4705,4832,4962,5035,5078,5145,5339,5841,5869,5519,5744,5988,5937,6013,6537,7606,8221,7440,5799,4816,5047,4725,4399,
    5299,4824,4552,4835,5247,5240,4969,4654,4584,4880,5103,4997,4756,4559,4750,5131,5115,5047,5044,5068,5112,5341,5527,5330,5671,6102,5718,5807,6073,6226,6426,6639,7234,7755,6964,5213,4522,4652,4699,4729,
    4993,4608,4811,4965,5297,5324,5039,4759,4738,4956,5088,4899,4748,4833,4943,5330,5355,5264,5249,5275,5263,5340,5508,5546,5754,6141,5848,5931,6110,6244,6164,6145,6571,7203,6924,5503,4439,4248,4253,4472,
    4751,4654,5041,5206,5232,5078,4875,4846,4960,5074,5139,4990,5074,5135,4979,5212,5441,5410,5628,5888,5975,6180,6545,6625,6605,6530,6027,6090,6099,6091,5925,6030,6250,6463,5967,4796,4416,4287,4104,4045,
    4954,4788,5125,5325,5219,5061,4895,4954,5171,5252,5282,5270,5383,5242,5109,5145,5478,5611,6338,6784,7078,7568,7589,6907,6248,6119,5827,5763,5724,5747,5652,5629,5554,5419,5186,4881,4815,4854,4551,4235,
    5048,4889,5152,5285,5184,5055,4958,5100,5344,5390,5418,5504,5445,5295,5205,5181,5402,5865,6405,6724,7277,7996,7866,6683,5697,5601,5543,5430,5373,5341,5272,5167,5023,4860,4748,4717,4694,4599,4434,4303,
    5001,4982,5216,5326,5260,5087,4989,5112,5314,5397,5456,5534,5519,5392,5360,5357,5349,5735,6202,6597,7137,7762,7770,6729,5640,5468,5405,5344,5346,5316,5258,5178,5062,4909,4743,4599,4487,4384,4430,4533,
    5121,5145,5220,5349,5333,5180,5035,5060,5220,5404,5563,5582,5797,5648,5714,5768,5546,5656,6031,6371,6809,7258,7287,6786,5951,5621,5465,5507,5549,5593,5560,5519,5425,5213,4945,4740,4609,4527,4728,4838,
    5353,5300,5234,5337,5354,5245,5123,5101,5191,5442,5806,5849,6053,5910,5967,5972,5860,5900,5894,6018,6469,6813,6716,6572,6327,5906,5746,5807,5825,5885,5886,5861,5725,5436,5092,4858,4766,4681,4934,5058,
    5416,5411,5403,5495,5453,5279,5255,5277,5264,5514,5998,6049,6029,5893,5844,5719,5900,6000,5845,6023,6420,6595,6514,6399,6550,6225,6071,6015,5978,6041,6076,6050,5868,5562,5238,4988,4881,4880,5110,5306,
    5390,5674,5739,5966,5722,5415,5478,5594,5584,5759,6044,5948,5892,5843,5769,5707,5692,5806,5763,6000,6234,6240,6315,6441,6604,6453,6173,6016,5856,5919,5942,5944,5882,5682,5369,5171,5215,5138,5265,5290,
    5641,6050,6360,6800,6375,5858,5858,5988,5966,6002,6091,6091,5950,6258,6132,6050,5774,5528,5678,6110,6248,6131,6289,6608,6645,6579,6261,6021,5878,5754,5729,5750,5671,5474,5209,5089,5307,5392,5118,4957,
    6189,6197,6309,6648,6795,6511,6451,6484,6397,6437,6452,6279,6156,6360,6354,6187,5932,5534,5678,5971,6002,6001,6186,6352,6402,6358,6247,6014,5964,5886,5739,5587,5443,5307,5143,4986,4934,4901,4947,4782,
    6075,6027,6439,7150,7413,7287,7114,6969,6826,6819,6835,6347,6246,6105,6184,6173,5931,5486,5534,5789,6040,6279,6398,6454,6524,6370,6205,6145,6084,5942,5742,5579,5475,5468,5378,5077,4831,4737,4728,4848,
    5773,5931,6176,6529,6477,6735,6726,6525,6397,6231,6248,6289,6244,6080,5878,6055,5729,5443,5755,6134,6335,6402,6385,6466,6502,6552,6357,6027,5856,5715,5637,5581,5572,5499,5370,5269,5162,5259,5369,5310,
    5812,6087,6303,6381,6235,6379,6451,6316,6135,6027,6150,6343,6209,6009,5737,5747,5423,5494,5921,6188,6145,6119,6269,6274,6051,6172,6055,5983,5962,5924,5782,5641,5586,5564,5494,5338,5207,5372,5385,5256,
    5676,5911,6032,6065,6183,6531,6625,6398,6130,5944,5918,6115,6183,6015,5872,5555,5289,5536,5817,5923,6023,6187,6255,6100,5893,6032,5806,5899,5830,5960,5902,5829,5689,5538,5418,5264,5167,5181,5148,5055,
    5502,5721,5942,6187,6320,6475,6497,6361,6180,5964,5793,5935,6135,5912,5719,5336,5327,5740,5878,5900,6011,6093,6067,5959,5786,5915,5650,5795,5933,5981,5836,5701,5554,5409,5316,5255,5236,5254,5249,5154,
    5472,5736,5978,6130,6268,6386,6346,6219,6111,5961,5760,5729,5948,5830,5482,5184,5505,5809,5922,5953,5915,5850,5822,5748,5648,5676,5570,5866,5912,5820,5636,5471,5372,5330,5322,5324,5328,5316,5196,5092,
    5481,5718,5842,6015,6156,6290,6325,6267,6176,6021,5768,5557,5746,5744,5259,5182,5556,5733,5833,5827,5696,5602,5586,5522,5502,5396,5639,5806,5708,5503,5381,5251,5145,5133,5187,5225,5223,5173,5034,4943,
    5383,5576,5664,5896,6055,6171,6257,6292,6233,6076,5835,5510,5556,5678,5113,5274,5609,5669,5648,5560,5462,5402,5357,5332,5264,5320,5634,5523,5374,5302,5359,5257,5066,4966,4964,4978,4974,4928,4846,4791,
    5220,5421,5557,5731,5905,6053,6162,6226,6206,6088,5880,5536,5340,5581,5054,5318,5668,5630,5439,5294,5282,5243,5126,5148,5117,5427,5467,5223,5073,5212,5394,5313,5125,5003,4937,4886,4847,4816,4773,4874,
    5090,5307,5499,5682,5829,5995,6125,6191,6209,6123,5872,5545,5166,5336,4988,5240,5526,5543,5298,5096,5095,5076,4932,4955,5144,5358,5191,4980,4946,5071,5167,5136,5083,5085,5107,5071,4972,4937,4934,5229,
    4999,5238,5562,5725,5912,6012,5982,6005,6095,6071,5884,5522,5120,5037,4942,5156,5343,5384,5205,4957,4873,4918,4870,4795,4984,5032,4780,4767,4847,4932,4967,5020,5084,5183,5303,5309,5174,5131,5118,5346,
    4958,5201,5473,5650,5885,5870,5795,5870,5935,5851,5727,5484,5187,4840,5013,5190,5208,5111,4938,4790,4814,4952,4925,4721,4804,4942,4668,4703,4736,4772,4870,5041,5299,5538,5597,5428,5201,5203,5294,5429,
    4931,5062,5309,5502,5803,5807,5654,5570,5563,5600,5636,5499,5188,4852,4855,4943,4911,4838,4720,4681,4777,4912,4923,4771,4707,4940,4653,4634,4785,4867,4934,5118,5372,5625,5674,5412,5122,5047,5350,5606,
    4954,4898,5073,5313,5702,5734,5520,5350,5293,5308,5322,5241,5114,4931,4726,4663,4657,4648,4597,4558,4568,4636,4728,4735,4620,4725,4511,4911,5098,5169,5227,5301,5512,5693,5618,5413,5229,4924,5347,5673
});

var grids = new[] { world, tibetanPlateau, himalayaArc, everestRegion };
var labels = new[] { "Monde", "Plateau", "Himalaya", "Everest" };
foreach (var g in grids)
{
    var (u, v, e) = g.GlobalMax();
    Console.WriteLine($"{g.Name,-16} {g.N}x{g.N}  sommet={e,5} m @ (u={u:F3}, v={v:F3})  bbox lat[{g.LatMin},{g.LatMax}] lon[{g.LonMin},{g.LonMax}]");
}
Console.WriteLine("\nLe sommet monte avec le zoom : le maximum global se resserre vers l'Everest.");

World            40x40  sommet= 5102 m @ (u=0,744, v=0,744)  bbox lat[-56,60] lon[-180,180]


TibetanPlateau   40x40  sommet= 6209 m @ (u=0,359, v=0,462)  bbox lat[27,45] lon[70,100]


HimalayaArc      40x40  sommet= 6558 m @ (u=0,846, v=0,205)  bbox lat[27,30,5] lon[83,89]


EverestRegion    40x40  sommet= 8221 m @ (u=0,846, v=0,333)  bbox lat[27,7,28,3] lon[86,6,87,2]



Le sommet monte avec le zoom : le maximum global se resserre vers l'Everest.


## 2. Visualiser les quatre zooms

Meme rendu ASCII grayscale que MGS-7, mais l'echelle est **inversee par rapport a MGS-7** :
ici on maximise, donc la **haute altitude** est le caractere dense `@` et les basses terres /
oceans sont clairsemes. Le sommet (maximum global) est marque `^`. Le nord (haute latitude)
est en haut.

In [4]:
const string Ramp = " .:-=+*#%@";   // sparse (low) -> dense (high elevation)

char Shade(double v, double vmin, double vmax)
{
    if (vmax - vmin < 1e-9) return Ramp[Ramp.Length - 1];
    double t = (v - vmin) / (vmax - vmin);                 // 0 = low, 1 = high
    int idx = (int)(t * (Ramp.Length - 1));                // high elevation -> dense '@'
    return Ramp[Math.Clamp(idx, 0, Ramp.Length - 1)];
}

void PrintRelief(DemGrid g, int wdisp = 60, int hdisp = 24)
{
    var (gu, gv, gmaxE) = g.GlobalMax();
    int mi = (int)Math.Round(gu * (wdisp - 1)), mj = (int)Math.Round(gv * (hdisp - 1));
    double vmin = double.MaxValue, vmax = double.MinValue;
    var z = new double[wdisp, hdisp];
    for (int i = 0; i < wdisp; i++)
        for (int j = 0; j < hdisp; j++)
        {
            double e = g.Interp((double)i / (wdisp - 1), (double)j / (hdisp - 1));
            z[i, j] = e; if (e < vmin) vmin = e; if (e > vmax) vmax = e;
        }
    Console.WriteLine($"=== {g.Name} : altitude {vmin:F0}..{vmax:F0} m   (^ = sommet {gmaxE} m) ===");
    for (int j = hdisp - 1; j >= 0; j--)                   // north (high lat) at top
    {
        var sb = new System.Text.StringBuilder();
        for (int i = 0; i < wdisp; i++)
            sb.Append((i == mi && j == mj) ? '^' : Shade(z[i, j], vmin, vmax));
        Console.WriteLine("  " + sb);
    }
}

foreach (var g in grids) { PrintRelief(g); Console.WriteLine(); }

=== World : altitude -7389..4923 m   (^ = sommet 5102 m) ===


  -=+++++++++++++++++=-------==+++++++++++++++++++++++++++++=-


  ===+-::=+*++++++++++=-::---==++++++++++++++++**+**+++++=---=


  :....:::-***++++++++++-::::-=++++++++++++++*****++++++=-..::


  ......::-=****++++++++-:--::::+++++++++++++++****+++++. .::.


...

  ==----::.......::-==+================+++++++==----------=---


  ==--==--::..  ..:-=+++++============++*++==========--::::---


  ===--:::..     ..:-=======--====---==++===------------::::::


  -===-:::.      ..:--=--:--=====-----======--:::::::::::::::.


  -----::...    .:--=--::-=====+==--::--==--::::::::::::....::


**Lecture.** Du Monde a l'Everest, deux choses changent ensemble :

- le **sommet s'eleve** (5102 m -> 6209 -> 6558 -> 8221 m) : au zoom grossier la maille moyenne
  les pics et le maximum n'est qu'un haut-plateau emousse ; au zoom fin l'Everest emerge ;
- le **bassin d'attraction se resserre** : sur la carte Monde la zone "haute" couvre une large
  fraction de l'image (facile a toucher au hasard), alors qu'a la region Everest le pic occupe
  une petite portion d'une **petite** boite. C'est ce double mouvement - pic plus net dans un
  espace plus etroit - qui rend le taux de reussite non trivial (section 4).

## 3. Chercher le sommet avec le moteur MGS

On encapsule le relief dans une `DemFitness : IFitness` (altitude interpolee = fitness). L'espace
de recherche est la boite normalisee `[0,1]^2` ; un chromosome porte `(u, v)`. On compare quatre
strategies a **budget d'evaluations egal** (NFE = population x generations) :

| Methode | Origine | Role |
|---------|---------|------|
| **GA**  | `DefaultMetaHeuristic` (moteur MGS) | reference evolutionnaire |
| **WOA** | `WhaleOptimisationAlgorithm` (compound geometrique du fork) | meta-heuristique assemblee |
| **EO**  | `EquilibriumOptimizer` (compound geometrique du fork) | meta-heuristique assemblee |
| **PSO** | baseline **externe** (canon Kennedy-Eberhart) | controle hors-bestiaire |

GA/WOA/EO passent par le **vrai moteur** `MetaGeneticAlgorithm` ; seul le PSO est un essaim
ecrit ici a la main, car le fork n'expose pas de PSO.

In [5]:
// Elevation as a fitness to MAXIMISE. Wrapping only: the relief data and the MGS engine
// are untouched. KnownFunctionGenes.AsDoubles reads the gene values (library helper).
public sealed class DemFitness : IFitness
{
    private readonly DemGrid _g;
    public DemFitness(DemGrid g) => _g = g;
    public double Evaluate(IChromosome c)
    {
        var x = KnownFunctionGenes.AsDoubles(c);
        return _g.Interp(x[0], x[1]);     // metres ; higher = better
    }
}

// DoubleArrayChromosome with per-gene bounds so CreateNew() seeds the initial population
// (same helper as MGS-5/MGS-7). Genes are the normalised coordinates (u, v) in [0,1].
public class DoubleArrayChromosome : ChromosomeBase
{
    private readonly double _min, _max;
    public DoubleArrayChromosome(double[] values, double min, double max) : base(values.Length)
    {
        _min = min; _max = max;
        for (int i = 0; i < values.Length; i++) ReplaceGene(i, new Gene(values[i]));
    }
    public override IChromosome CreateNew()
    {
        var rand = RandomizationProvider.Current;
        var vals = new double[Length];
        for (int i = 0; i < Length; i++) vals[i] = rand.GetDouble(_min, _max);
        return new DoubleArrayChromosome(vals, _min, _max);
    }
    public override Gene GenerateGene(int geneIndex) => new Gene(RandomizationProvider.Current.GetDouble(_min, _max));
    public double[] GetDoubleValues() => GetGenes().Select(g => (double)g.Value).ToArray();
}

// Deterministic randomization so each (method, zoom, seed) run is reproducible.
public sealed class SeededRandomization : RandomizationBase
{
    private readonly Random _r;
    public SeededRandomization(int seed) => _r = new Random(seed);
    public override int GetInt(int min, int max) => _r.Next(min, max);
    public override float GetFloat() => (float)_r.NextDouble();
    public override double GetDouble() => _r.NextDouble();
}

Console.WriteLine("DemFitness + DoubleArrayChromosome + SeededRandomization definis.");

DemFitness + DoubleArrayChromosome + SeededRandomization definis.


In [6]:
const int Pop = 30, Gens = 30;   // shared budget : NFE = Pop * Gens = 900 evaluations.

// Run the REAL MGS engine on DemFitness with a given metaheuristic ; return best (u,v,elev).
(double u, double v, double elev) RunEngine(DemGrid g, Func<IMetaHeuristic> buildMh, int seed)
{
    RandomizationProvider.Current = new SeededRandomization(seed);
    var fit = new DemFitness(g);
    var adam = new DoubleArrayChromosome(new[] { 0.5, 0.5 }, 0.0, 1.0);
    var pop = new MetaPopulation(Pop, Pop, adam);
    var ga = new MetaGeneticAlgorithm(pop, fit, new EliteSelection(),
        new UniformCrossover(0.5f), new UniformMutation(true), buildMh());
    ga.Termination = new GenerationNumberTermination(Gens);
    ga.Start();
    // MetaPopulation is order-preserving: derive best via fitness, not CurrentGeneration.BestChromosome.
    var best = ga.Population.CurrentGeneration.Chromosomes.OrderByDescending(c => c.Fitness).First();
    var x = ((DoubleArrayChromosome)best).GetDoubleValues();
    double u = Math.Clamp(x[0], 0, 1), v = Math.Clamp(x[1], 0, 1);
    return (u, v, g.Interp(u, v));
}

IMetaHeuristic BuildGA() => new DefaultMetaHeuristic();

IMetaHeuristic BuildWOA()
{
    var woa = new WhaleOptimisationAlgorithm { MaxGenerations = Gens };
    woa.SetGeometricConverter(new GeometricConverter<double> {
        GeneToDoubleConverter = (_, val) => val, DoubleToGeneConverter = (_, d) => d });
    return woa.Build();
}

IMetaHeuristic BuildEO()
{
    var eo = new EquilibriumOptimizer { MaxGenerations = Gens };
    eo.SetGeometricConverter(new GeometricConverter<double> {
        GeneToDoubleConverter = (_, val) => val, DoubleToGeneConverter = (_, d) => d });
    return eo.Build();
}

// PSO : explicit EXTERNAL baseline (no PSO in the fork). Canonical Kennedy-Eberhart swarm on
// the same DemFitness, same NFE budget (swarm * iters = Pop * Gens).
(double u, double v, double elev) RunPSO(DemGrid g, int seed)
{
    var rng = new Random(seed);
    int swarm = Pop, iters = Gens;
    double w = 0.7, c1 = 1.5, c2 = 1.5;
    var px = new double[swarm]; var py = new double[swarm];
    var vx = new double[swarm]; var vy = new double[swarm];
    var bx = new double[swarm]; var by = new double[swarm]; var bf = new double[swarm];
    double gx = 0, gy = 0, gf = double.NegativeInfinity;
    for (int i = 0; i < swarm; i++)
    {
        px[i] = rng.NextDouble(); py[i] = rng.NextDouble();
        vx[i] = (rng.NextDouble() - 0.5) * 0.1; vy[i] = (rng.NextDouble() - 0.5) * 0.1;
        bx[i] = px[i]; by[i] = py[i]; bf[i] = g.Interp(px[i], py[i]);
        if (bf[i] > gf) { gf = bf[i]; gx = px[i]; gy = py[i]; }
    }
    for (int t = 1; t < iters; t++)
        for (int i = 0; i < swarm; i++)
        {
            vx[i] = w * vx[i] + c1 * rng.NextDouble() * (bx[i] - px[i]) + c2 * rng.NextDouble() * (gx - px[i]);
            vy[i] = w * vy[i] + c1 * rng.NextDouble() * (by[i] - py[i]) + c2 * rng.NextDouble() * (gy - py[i]);
            px[i] = Math.Clamp(px[i] + vx[i], 0, 1); py[i] = Math.Clamp(py[i] + vy[i], 0, 1);
            double f = g.Interp(px[i], py[i]);
            if (f > bf[i]) { bf[i] = f; bx[i] = px[i]; by[i] = py[i]; }
            if (f > gf) { gf = f; gx = px[i]; gy = py[i]; }
        }
    return (gx, gy, gf);
}

var sm = RunEngine(everestRegion, BuildWOA, 0);
var (tu, tv, te) = everestRegion.GlobalMax();
Console.WriteLine($"Smoke WOA @ EverestRegion : best=({sm.u:F3},{sm.v:F3}) elev={sm.elev:F0} m  |  cible=({tu:F3},{tv:F3}) {te} m");

Smoke WOA @ EverestRegion : best=(0,538,0,470) elev=7919 m  |  cible=(0,846,0,333) 8221 m


## 4. Taux de reussite x 4 zooms

On declare une recherche **reussie** si le meilleur point trouve tombe a une distance normalisee
`<= tau` de la cellule du maximum global (rayon `tau = 0.06`, soit ~6% du cote de la boite). Pour
chaque (methode, zoom) on lance `Seeds` graines independantes et on rapporte le pourcentage de
reussites - a budget d'evaluations **identique** pour les quatre methodes.

In [7]:
const double Tau = 0.06;     // success radius : ~6% of the box side
const int Seeds = 10;

bool Hit(DemGrid g, double u, double v)
{
    var (gu, gv, _) = g.GlobalMax();
    double du = u - gu, dv = v - gv;
    return Math.Sqrt(du * du + dv * dv) <= Tau;
}

double Rate(DemGrid g, Func<DemGrid, int, (double u, double v, double elev)> run)
{
    int hits = 0;
    for (int s = 0; s < Seeds; s++) { var r = run(g, s); if (Hit(g, r.u, r.v)) hits++; }
    return 100.0 * hits / Seeds;
}

var methods = new (string name, Func<DemGrid, int, (double u, double v, double elev)> run)[]
{
    ("GA",  (g, s) => RunEngine(g, BuildGA,  s)),
    ("PSO", (g, s) => RunPSO(g, s)),
    ("WOA", (g, s) => RunEngine(g, BuildWOA, s)),
    ("EO",  (g, s) => RunEngine(g, BuildEO,  s)),
};

Console.WriteLine($"Taux de reussite (%) -- {Seeds} graines, budget NFE = {Pop * Gens}, tau = {Tau}\n");
Console.WriteLine(string.Format("{0,-6}{1,10}{2,10}{3,10}{4,10}", "methode", "Monde", "Plateau", "Himalaya", "Everest"));
Console.WriteLine(new string('-', 46));
foreach (var m in methods)
{
    var sb = new System.Text.StringBuilder(string.Format("{0,-6}", m.name));
    foreach (var g in grids) sb.Append(string.Format("{0,9:F0}%", Rate(g, m.run)));
    Console.WriteLine(sb.ToString());
}

Taux de reussite (%) -- 10 graines, budget NFE = 900, tau = 0,06



methode     Monde   Plateau  Himalaya   Everest


----------------------------------------------


GA           80%       20%        0%       10%


PSO         100%       60%       10%       30%


WOA          60%       40%        0%       10%


EO           80%       70%        0%       10%


**Lecture (No Free Lunch).** Les chiffres ci-dessus sont produits a l'execution - on les lit
sans les maquiller :

- **Aucune methode ne domine partout.** Sur ce relief et a ce budget, c'est meme le **PSO**
  (baseline *externe*, hors-bestiaire) qui mene plusieurs colonnes : l'assemblage geometrique
  WOA/EO n'a pas d'avantage decisif sur un essaim simple. La sophistication n'est pas gratuite -
  signature du theoreme **No Free Lunch**.
- **Le taux n'est pas monotone en zoom.** Un pic plus net (Everest) est plus dur a *localiser*
  precisement, mais il vit dans une boite plus petite ou le meme `tau` couvre proportionnellement
  plus de terrain : les deux effets se compensent differemment selon la methode.
- **Certaines colonnes frolent 0 %.** Le maximum de la carte Himalaya tombe pres d'un **bord**
  (visible sur le relief de la section 2) et `tau = 0.06` est serre : le bassin utile est minuscule,
  toutes methodes peinent. C'est un cas-limite honnete, pas un bug - le smoke-test de la section 3
  confirme que le moteur trouve bien de la haute altitude.

## 5. Exercices

Trois exercices completables **hors-ligne** (aucune donnee supplementaire requise). Decommentez
et completez ; chaque cellule s'execute deja telle quelle (stub conforme).

In [8]:
// Exercice 1 : sensibilite a la tolerance tau.
// Le taux de reussite depend du rayon d'acceptation. Re-tabulez les 4 methodes x 4 zooms
// pour tau dans { 0.03, 0.06, 0.12 } et observez comment la hierarchie des methodes bouge.
// TODO etudiant :
//   foreach (double tau in new[] { 0.03, 0.06, 0.12 }) {
//       // Indice : copiez Hit/Rate en parametrant tau, puis ré-affichez la table.
//   }

Console.WriteLine("Exercice a completer : taux de reussite en fonction de tau (0.03 / 0.06 / 0.12).");

Exercice a completer : taux de reussite en fonction de tau (0.03 / 0.06 / 0.12).


In [9]:
// Exercice 2 : ajouter une baseline "hill-climber" (montee de gradient stochastique).
// Une 5e ligne au tableau : depart aleatoire, petits pas (u,v) +- epsilon, on garde le pas
// s'il monte. Meme budget NFE = Pop * Gens. Ou se classe-t-il face a GA/PSO/WOA/EO ?
// TODO etudiant :
//   (double u, double v, double elev) RunHillClimber(DemGrid g, int seed) {
//       // Indice : 1 point courant, NFE = Pop*Gens evaluations, pas epsilon ~ 0.05, clamp [0,1].
//       return (0, 0, 0); // remplacer
//   }

Console.WriteLine("Exercice a completer : baseline hill-climber, meme budget NFE.");

Exercice a completer : baseline hill-climber, meme budget NFE.


In [10]:
// Exercice 3 : courbe taux de reussite vs budget, sur la region Everest.
// Faites varier le budget NFE (par ex. Pop*Gens dans { 200, 450, 900, 1800 }) pour la WOA
// sur everestRegion et tracez (en ASCII ou en colonne) le taux de reussite correspondant.
// TODO etudiant :
//   foreach (int nfe in new[] { 200, 450, 900, 1800 }) {
//       // Indice : ajustez Gens = nfe / Pop, relancez Rate(everestRegion, ...), affichez.
//   }

Console.WriteLine("Exercice a completer : taux de reussite vs budget NFE (region Everest).");

Exercice a completer : taux de reussite vs budget NFE (region Everest).


## Conclusion

| Element | Ce que MGS-8 montre |
|---------|---------------------|
| Relief reel comme fitness | une grille ETOPO1 embarquee, interpolee, **maximisee** (trouver l'Everest) |
| Rendu | ASCII grayscale coherent avec MGS-7, mais haute altitude = dense `@` |
| Moteur | GA / WOA / EO via le **vrai** `MetaGeneticAlgorithm` + compounds geometriques |
| Baseline | PSO externe explicite (le fork n'a pas de PSO) - controle a budget egal |
| Lecon | **bassin d'attraction vs taille de l'espace** : le sommet se resserre avec le zoom,
  et aucune methode ne gagne partout (**No Free Lunch**) |

**Pour aller plus loin** : [MGS-5 - Benchmarks](MGS-5-Benchmarks.ipynb) (WOA vs Islands sur
fonctions analytiques), [MGS-7 - Landscape Explorer](MGS-7-LandscapeExplorer.ipynb) (paysages
synthetiques et biais de centre).

*Donnees : NOAA NCEI ETOPO1 1 arc-minute global relief, downsamplees 40x40 par zoom et embarquees.*